In [1]:
%useLatestDescriptors
%use dataframe, kandy
%use kandy-geo

In [92]:
USE {
    dependencies {
        implementation("org.xerial:sqlite-jdbc:3.49.1.0")
        implementation("ch.qos.logback:logback-classic:1.5.12")
    }
}

In [3]:
import java.sql.Connection
import java.sql.DriverManager

In [36]:
Class.forName("org.sqlite.JDBC")
val connection = DriverManager.getConnection("jdbc:sqlite:/Users/unchil/full-stack-task-manager/full-stack-task-manager.sqlite")

In [4]:
import java.time.*
import java.time.format.DateTimeFormatter
import java.text.SimpleDateFormat

In [34]:
val formatter = DateTimeFormatter.ofPattern("YYYY-MM-dd HH:mm:ss")
val now = java.time.LocalDateTime.now()
val prevDay = now.minusHours(24)

print("Current time : ${now.format(formatter)}, Previous time : ${prevDay.format(formatter)}")

Current time : 2025-03-17 10:42:59, Previous time : 2025-03-16 10:42:59

In [47]:
val whereStmt_last24h  = "WHERE obs_datetime > '${prevDay.format(formatter)}' "
val sqlStmt = "SELECT * FROM Observation " + whereStmt_last24h
val df_list = DataFrame.readSqlQuery(connection, sqlStmt)
df_list.describe()

name,type,count,unique,nulls,top,freq,min,median,max
sta_cde,String,2965,44,0,bgj8a,126,bgj8a,fnm5b,wn087
sta_nam_kor,String,2965,44,0,기장,126,강릉,완도 가교,해남 화산
obs_dat,String,2965,2,0,2025-03-17,1555,2025-03-16,2025-03-17,2025-03-17
obs_tim,String,2965,43,0,14:00:00,71,00:00:00,10:00:00,23:30:00
repair_gbn,String,2965,1,0,1,2965,1,1,1
obs_lay,String,2965,3,0,1,1837,1,1,3
wtr_tmp,String,2965,94,0,8.9,139,10,7.8,9.9
dox,String?,2965,44,2339,9.6,46,10,12.2,9.9
sal,String?,2965,5,2881,32.5,42,32.5,32.5,34.9
obs_datetime,String,2965,43,0,2025-03-16 14:00:00,71,2025-03-16 13:30:00,2025-03-17 00:30:00,2025-03-17 10:30:00


In [48]:
val df_code = DataFrame.readSqlTable(connection, "Observatory")
df_code.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
sta_cde,String,84,84,0,fwdo5,1,null,null,bgj8a,fwdo5,ys002
sta_nam_kor,String,84,80,0,여수,2,null,null,강릉,영광,흑산도 대둔
bld_dat,String,84,64,0,2010-03-14,7,null,null,2003-11-25,2008-07-23,2024-05-13
end_dat,String?,84,38,44,2010-03-15,3,null,null,2005-01-20,2010-03-15,2024-05-25
gru_nam,String,84,3,0,남해,50,null,null,남해,남해,서해
lon,Double,84,84,0,126.736400,1,127.433104,1.137047,124.729500,127.236850,129.813100
lat,Double,84,84,0,34.382500,1,35.182755,1.251893,33.291000,34.741435,38.368100
sur_tmp_yn,String,84,2,0,Y,53,null,null,N,Y,Y
mid_tmp_yn,String,84,2,0,N,61,null,null,N,N,Y
bot_tmp_yn,String,84,2,0,N,75,null,null,N,N,Y


In [49]:
val df = df_list.innerJoinWith(df_code.select { sta_cde and gru_nam and lon and lat }) {
    right.sta_cde == sta_cde
}
df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
sta_cde,String,2965,44,0,bgj8a,126,null,null,bgj8a,fnm5b,wn087
sta_nam_kor,String,2965,44,0,기장,126,null,null,강릉,완도 가교,해남 화산
obs_dat,String,2965,2,0,2025-03-17,1555,null,null,2025-03-16,2025-03-17,2025-03-17
obs_tim,String,2965,43,0,14:00:00,71,null,null,00:00:00,10:00:00,23:30:00
repair_gbn,String,2965,1,0,1,2965,null,null,1,1,1
obs_lay,String,2965,3,0,1,1837,null,null,1,1,3
wtr_tmp,String,2965,94,0,8.9,139,null,null,10,7.8,9.9
dox,String?,2965,44,2339,9.6,46,null,null,10,12.2,9.9
sal,String?,2965,5,2881,32.5,42,null,null,32.5,32.5,34.9
obs_datetime,String,2965,43,0,2025-03-16 14:00:00,71,null,null,2025-03-16 13:30:00,2025-03-17 00:30:00,2025-03-17 10:30:00


In [59]:
df.filter{ gru_nam.equals("동해") and obs_lay.equals("1")  }
    .select{ sta_cde and sta_nam_kor and wtr_tmp }
    .convert { wtr_tmp }.with{ it.toFloat()}
    .groupBy{sta_cde and sta_nam_kor}
    .sortBy { sta_cde and  sta_nam_kor }
    .plot{
        layout {
            title = "관측지점별 일평균 표층 해수 정보"
            size = 1000 to 600
            //    theme = Theme.HIGH_CONTRAST_DARK
            x.axis {
                name = "관측지점"
            }
            y.axis {
                name = "온도"
                limits = 0.0..20.0
            }
        }

        boxplot("sta_nam_kor", "wtr_tmp") {
            boxes {
                borderLine.color = Color.BLUE
                fillColor("sta_nam_kor"){
                    scale = categoricalColorHue()
                    legend{
                        name= "관측 지점"
                    }
                }
            }
        }


    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="2tO65d"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일평균 표층 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[0.0,20.0]
},
"data":{
},
"ggsize":{
"width":1000.0,
"height":600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sta_nam_kor",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"wtr_tmp",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_hue",
"name":"관측 지점"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"name":"관측지점",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"온도",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max",
"fill":"sta_nam_kor",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["bgj8a$기장","bgna3$강릉","bsc87$삼척","byd8a$영덕","byy87$양양","fggo3$고성 가진","fghe8$구룡포 하정"],
"min":[11.600000381469727,5.0,6.699999809265137,10.899999618530273,6.599999904632568,5.5,10.0],
"middle":[11.899999618530273,5.599999904632568,7.099999904632568,11.300000190734863,6.900000095367432,6.099999904632568,10.449999809265137],
"max":[12.300000190734863,6.400000095367432,7.599999904632568,11.399999618530273,7.300000190734863,7.0,11.0],
"lower":[11.800000190734863,5.450000047683716,6.900000095367432,11.0,6.800000190734863,5.800000190734863,10.199999809265137],
"upper":[12.100000381469727,6.199999809265137,7.300000190734863,11.399999618530273,7.199999809265137,6.400000095367432,10.924999713897705],
"x":["기장","강릉","삼척","영덕","양양","고성 가진","구룡포 하정"],
"sta_nam_kor":["기장","강릉","삼척","영덕","양양","고성 가진","구룡포 하정"]
},
"color":"#5470c6",
"sampling":"none",
"inherit_aes":false,
"position":{
"name":"dodge",
"width":1.0
},
"geom":"boxplot",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_cde"
},{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"lower"
},{
"type":"float",
"column":"middle"
},{
"type":"float",
"column":"upper"
},{
"type":"float",
"column":"max"
},{
"type":"str",
"column":"&merged_groups"
}]
}
},{
"mapping":{
"x":"x",
"y":"y",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":[]
},
"sampling":"none",
"inherit_aes":false,
"position":{
"name":"dodge",
"width":1.0
},
"geom":"point"
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_cde"
},{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"str",
"column":"&merged_groups"
}]
},
"spec_id":"32"
};
 var containerDiv = document.getElementById("2tO65d");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1000.0,
 height: 600.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 기장 
 
 
 
 
 
 
 
 
 강릉 
 
 
 
 
 
 
 
 
 삼척 
 
 
 
 
 
 
 
 
 영덕 
 
 
 
 
 
 
 
 
 양양 
 
 
 
 
 
 
 
 
 고성 가진 
 
 
 
 
 
 
 
 
 구룡포 하정 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 14 
 
 
 
 
 
 

In [42]:
df.columnNames()

[sta_cde, sta_nam_kor, obs_dat, obs_tim, repair_gbn, obs_lay, wtr_tmp, dox, sal, obs_datetime, sta_cde1, gru_nam, lon, lat]

In [60]:
val df_East = df.filter {
    gru_nam.equals("동해") and obs_lay.equals("1")
}.add {
    "hour" from obs_tim.map {
        LocalTime.parse(it).hour
    }
}.select{
    sta_nam_kor and wtr_tmp and obs_tim  and gru_nam and sta_cde and obs_datetime and "hour"
}.convert { wtr_tmp }.with{ it.toFloat()}


df_East

sta_nam_kor,wtr_tmp,obs_tim,gru_nam,sta_cde,obs_datetime,hour
기장,12.300000,14:00:00,동해,bgj8a,2025-03-16 14:00:00,14
강릉,5.500000,14:00:00,동해,bgna3,2025-03-16 14:00:00,14
삼척,7.400000,14:00:00,동해,bsc87,2025-03-16 14:00:00,14
영덕,11.300000,14:00:00,동해,byd8a,2025-03-16 14:00:00,14
양양,6.900000,14:00:00,동해,byy87,2025-03-16 14:00:00,14
고성 가진,6.400000,14:00:00,동해,fggo3,2025-03-16 14:00:00,14
구룡포 하정,11.000000,14:00:00,동해,fghe8,2025-03-16 14:00:00,14
기장,12.100000,14:30:00,동해,bgj8a,2025-03-16 14:30:00,14
강릉,5.500000,14:30:00,동해,bgna3,2025-03-16 14:30:00,14
삼척,7.300000,14:30:00,동해,bsc87,2025-03-16 14:30:00,14


In [61]:
df_East
    .groupBy { sta_cde and sta_nam_kor and hour }
    .aggregate {
        min{wtr_tmp} into "min"
        max{wtr_tmp} into "max"
        mean{wtr_tmp} into "mean"
        min{obs_datetime} into "time"
    }
    .plot{
        layout {
            title = "관측지점별 일별 해수 정보"
            size = 2600 to 1200
        }

        ribbon {
            x("time") {
                axis.name = "측정시간"
            }

            yMin("min")
            yMax("max")

            alpha = 0.6
            borderLine.width = 0.0
            fillColor("sta_nam_kor"){
                legend {
                    name = "관측지점"
                }
            }

            tooltips(title = value(sta_nam_kor)){
                line("최저 ${value("min")}, 최고 ${value("max")}")
                line("측정시간", hour.tooltipValue("d"))
            }
        }

        line {
            x("time")
            y("mean")
            width = 1.0
            color = Color.BLUE
            tooltips(enable=false)
        }

        y.axis {
            limits = 4.0..12.0
            name = "수온 최저~최고"
        }

        facetWrap(nCol = 2) {
            facet(sta_nam_kor)
        }
    }


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="P7a4ep"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 일별 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[4.0,12.0]
},
"data":{
"min":[12.100000381469727,5.5,7.300000190734863,11.300000190734863,6.900000095367432,6.400000095367432,11.0,12.0,5.5,7.199999809265137,11.300000190734863,6.900000095367432,6.599999904632568,11.0,12.199999809265137,5.199999809265137,7.0,11.399999618530273,6.800000190734863,6.5,11.0,12.100000381469727,5.0,7.099999904632568,11.399999618530273,6.900000095367432,6.400000095367432,11.0,11.899999618530273,5.099999904632568,7.0,11.399999618530273,7.0,6.400000095367432,11.0,12.100000381469727,5.300000190734863,6.900000095367432,11.399999618530273,7.199999809265137,6.5,10.899999618530273,12.100000381469727,5.5,6.800000190734863,11.399999618530273,7.199999809265137,6.199999809265137,10.800000190734863,12.100000381469727,6.699999809265137,11.399999618530273,7.199999809265137,6.099999904632568,10.699999809265137,5.599999904632568,11.800000190734863,5.699999809265137,6.800000190734863,11.399999618530273,7.199999809265137,6.199999809265137,10.600000381469727,11.899999618530273,5.5,6.800000190734863,11.300000190734863,7.0,6.099999904632568,10.5,11.800000190734863,5.400000095367432,6.699999809265137,11.300000190734863,6.900000095367432,6.099999904632568,10.399999618530273,11.800000190734863,5.400000095367432,6.900000095367432,11.199999809265137,6.800000190734863,6.0,10.300000190734863,11.800000190734863,5.599999904632568,7.199999809265137,11.100000381469727,6.599999904632568,5.900000095367432,10.199999809265137,11.699999809265137,5.699999809265137,7.199999809265137,11.0,6.599999904632568,5.800000190734863,10.100000381469727,11.800000190734863,6.0,6.900000095367432,11.0,7.099999904632568,6.0,10.0,11.800000190734863,6.199999809265137,7.099999904632568,10.899999618530273,7.199999809265137,5.800000190734863,10.100000381469727,11.800000190734863,6.300000190734863,7.300000190734863,11.0,7.099999904632568,5.699999809265137,10.199999809265137,11.800000190734863,6.400000095367432,7.300000190734863,11.0,6.900000095367432,5.699999809265137,10.199999809265137,11.899999618530273,6.300000190734863,7.400000095367432,11.0,6.800000190734863,5.699999809265137,10.199999809265137,11.899999618530273,6.199999809265137,7.599999904632568,10.899999618530273,6.800000190734863,5.800000190734863,10.300000190734863,11.600000381469727,6.300000190734863,7.599999904632568,10.899999618530273,6.699999809265137,5.5,10.399999618530273],
"hour":[14.0,14.0,14.0,14.0,14.0,14.0,14.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0,18.0,18.0,18.0,18.0,18.0,18.0,18.0,19.0,19.0,19.0,19.0,19.0,19.0,19.0,20.0,20.0,20.0,20.0,20.0,20.0,20.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,22.0,22.0,22.0,22.0,22.0,22.0,22.0,23.0,23.0,23.0,23.0,23.0,23.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0],
"max":[12.300000190734863,5.5,7.400000095367432,11.300000190734863,6.900000095367432,6.5,11.0,12.100000381469727,5.5,7.199999809265137,11.399999618530273,6.900000095367432,6.900000095367432,11.0,12.199999809265137,5.400000095367432,7.099999904632568,11.399999618530273,6.900000095367432,7.0,11.0,12.199999809265137,5.300000190734863,7.099999904632568,11.399999618530273,6.900000095367432,6.5,11.0,12.0,5.199999809265137,7.099999904632568,11.39999961853027

In [62]:
df_East
    .select{  sta_nam_kor and wtr_tmp and obs_datetime   }
   // .convert{ obs_tim }.with{ LocalTime.parse(it) }
    .plot{

        layout {
            title = "동해 해수 정보"
            size = 2600 to 600
        }

        x(obs_datetime) { axis.name = "관측일시"}
        y(wtr_tmp) {axis.name ="표층수온"}
        y.axis.limits = 2.0..15.0
        line{
            color(sta_nam_kor){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측소명"
                }
            }
        }

        //  facetWrap(nRow = 3){ facet(gruNam)   }

    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="avKEHW"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"동해 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[2.0,15.0]
},
"data":{
"obs_datetime":["2025-03-16 14:00:00","2025-03-16 14:00:00","2025-03-16 14:00:00","2025-03-16 14:00:00","2025-03-16 14:00:00","2025-03-16 14:00:00","2025-03-16 14:00:00","2025-03-16 14:30:00","2025-03-16 14:30:00","2025-03-16 14:30:00","2025-03-16 14:30:00","2025-03-16 14:30:00","2025-03-16 14:30:00","2025-03-16 14:30:00","2025-03-16 15:00:00","2025-03-16 15:00:00","2025-03-16 15:00:00","2025-03-16 15:00:00","2025-03-16 15:00:00","2025-03-16 15:00:00","2025-03-16 15:00:00","2025-03-16 15:30:00","2025-03-16 15:30:00","2025-03-16 15:30:00","2025-03-16 15:30:00","2025-03-16 15:30:00","2025-03-16 15:30:00","2025-03-16 15:30:00","2025-03-16 16:00:00","2025-03-16 16:00:00","2025-03-16 16:00:00","2025-03-16 16:00:00","2025-03-16 16:00:00","2025-03-16 16:00:00","2025-03-16 16:00:00","2025-03-16 16:30:00","2025-03-16 16:30:00","2025-03-16 16:30:00","2025-03-16 16:30:00","2025-03-16 16:30:00","2025-03-16 16:30:00","2025-03-16 16:30:00","2025-03-16 17:00:00","2025-03-16 17:00:00","2025-03-16 17:00:00","2025-03-16 17:00:00","2025-03-16 17:00:00","2025-03-16 17:00:00","2025-03-16 17:00:00","2025-03-16 17:30:00","2025-03-16 17:30:00","2025-03-16 17:30:00","2025-03-16 17:30:00","2025-03-16 17:30:00","2025-03-16 17:30:00","2025-03-16 17:30:00","2025-03-16 18:00:00","2025-03-16 18:00:00","2025-03-16 18:00:00","2025-03-16 18:00:00","2025-03-16 18:00:00","2025-03-16 18:00:00","2025-03-16 18:00:00","2025-03-16 18:30:00","2025-03-16 18:30:00","2025-03-16 18:30:00","2025-03-16 18:30:00","2025-03-16 18:30:00","2025-03-16 18:30:00","2025-03-16 18:30:00","2025-03-16 19:00:00","2025-03-16 19:00:00","2025-03-16 19:00:00","2025-03-16 19:00:00","2025-03-16 19:00:00","2025-03-16 19:00:00","2025-03-16 19:00:00","2025-03-16 19:30:00","2025-03-16 19:30:00","2025-03-16 19:30:00","2025-03-16 19:30:00","2025-03-16 19:30:00","2025-03-16 19:30:00","2025-03-16 19:30:00","2025-03-16 20:00:00","2025-03-16 20:00:00","2025-03-16 20:00:00","2025-03-16 20:00:00","2025-03-16 20:00:00","2025-03-16 20:00:00","2025-03-16 20:00:00","2025-03-16 20:30:00","2025-03-16 20:30:00","2025-03-16 20:30:00","2025-03-16 20:30:00","2025-03-16 20:30:00","2025-03-16 20:30:00","2025-03-16 20:30:00","2025-03-16 21:00:00","2025-03-16 21:00:00","2025-03-16 21:00:00","2025-03-16 21:00:00","2025-03-16 21:00:00","2025-03-16 21:00:00","2025-03-16 21:30:00","2025-03-16 21:30:00","2025-03-16 21:30:00","2025-03-16 21:30:00","2025-03-16 21:30:00","2025-03-16 21:30:00","2025-03-16 21:30:00","2025-03-16 22:00:00","2025-03-16 22:00:00","2025-03-16 22:00:00","2025-03-16 22:00:00","2025-03-16 22:00:00","2025-03-16 22:00:00","2025-03-16 22:00:00","2025-03-16 22:30:00","2025-03-16 22:30:00","2025-03-16 22:30:00","2025-03-16 22:30:00","2025-03-16 22:30:00","2025-03-16 22:30:00","2025-03-16 22:30:00","2025-03-16 23:00:00","2025-03-16 23:00:00","2025-03-16 23:00:00","2025-03-16 23:00:00","2025-03-16 23:00:00","2025-03-16 23:00:00","2025-03-16 23:00:00","2025-03-16 23:30:00","2025-03-16 23:30:00","2025-03-16 23:30:00","2025-03-16 23:30:00","2025-03-16 23:30:00","2025-03-16 23:30:00","2025-03-16 23:30:00","2025-03-17 00:00:00","2025-03-17 00:00:00","2025-03-17 00:00:00","2025-03-17 00:00:00","2025-03-17 00:00:00","2025-03-17 00:00:00","2025-03-17 00:00:00","2025-03-17 00:30:00","2025-03-17 00:30:00","2025-03-17 00:30:00","2025-03-17 00:30:00","2025-03-17 00:30:00","2025-03-17 00:30:00","2025-03-17 00:30:00","2025-03-17 01:00:00","2025-03-17 01:00:00","2025-03-17 01:00:00","2025-03-17 01

In [63]:
df.obs_datetime.describe()

name,type,count,unique,nulls,top,freq,min,median,max
obs_datetime,String,2965,43,0,2025-03-16 14:00:00,71,2025-03-16 13:30:00,2025-03-17 00:30:00,2025-03-17 10:30:00


In [66]:
val currentTime = df.obs_datetime.max()

df.filter { obs_datetime.equals(currentTime) }

sta_cde,sta_nam_kor,obs_dat,obs_tim,repair_gbn,obs_lay,wtr_tmp,dox,sal,obs_datetime,sta_cde1,gru_nam,lon,lat
bgj8a,기장,2025-03-17,10:30:00,1,1,11.7,null,null,2025-03-17 10:30:00,bgj8a,동해,129.227000,35.187000
bgj8a,기장,2025-03-17,10:30:00,1,2,11.8,null,null,2025-03-17 10:30:00,bgj8a,동해,129.227000,35.187000
bgj8a,기장,2025-03-17,10:30:00,1,3,11.8,null,null,2025-03-17 10:30:00,bgj8a,동해,129.227000,35.187000
bgna3,강릉,2025-03-17,10:30:00,1,1,6.3,null,null,2025-03-17 10:30:00,bgna3,동해,128.949200,37.799000
bgna3,강릉,2025-03-17,10:30:00,1,2,6.3,null,null,2025-03-17 10:30:00,bgna3,동해,128.949200,37.799000
bgna3,강릉,2025-03-17,10:30:00,1,3,6.2,null,null,2025-03-17 10:30:00,bgna3,동해,128.949200,37.799000
br001,태안 고남,2025-03-17,10:30:00,1,1,5.8,null,null,2025-03-17 10:30:00,br001,서해,126.433300,36.415800
bsc87,삼척,2025-03-17,10:30:00,1,1,7.6,null,null,2025-03-17 10:30:00,bsc87,동해,129.312700,37.302300
bsc87,삼척,2025-03-17,10:30:00,1,2,7.5,null,null,2025-03-17 10:30:00,bsc87,동해,129.312700,37.302300
bsc87,삼척,2025-03-17,10:30:00,1,3,6.9,null,null,2025-03-17 10:30:00,bsc87,동해,129.312700,37.302300


In [82]:
import kotlinx.datetime.LocalTime

val df_Current = df.filter {
    obs_datetime.equals(currentTime)   and repair_gbn.equals("1")
  //  gru_nam.equals("동해")
}.select{
    sta_nam_kor and obs_lay and wtr_tmp and gru_nam and    sta_cde  and  lon and lat and obs_datetime
}.convert { wtr_tmp }.with{ it.toFloat()
}.sortBy{sta_nam_kor  and obs_lay }

df_Current

sta_nam_kor,obs_lay,wtr_tmp,gru_nam,sta_cde,lon,lat,obs_datetime
강릉,1,6.300000,동해,bgna3,128.949200,37.799000,2025-03-17 10:30:00
강릉,2,6.300000,동해,bgna3,128.949200,37.799000,2025-03-17 10:30:00
강릉,3,6.200000,동해,bgna3,128.949200,37.799000,2025-03-17 10:30:00
거제 가배,1,9.100000,남해,fgg4c,128.566400,34.785100,2025-03-17 10:30:00
거제 일운,1,10.700000,남해,gi086,128.709400,34.803800,2025-03-17 10:30:00
고성 가진,1,5.500000,동해,fggo3,128.523900,38.368100,2025-03-17 10:30:00
고성 가진,2,5.200000,동해,fggo3,128.523900,38.368100,2025-03-17 10:30:00
고성 가진,3,5.100000,동해,fggo3,128.523900,38.368100,2025-03-17 10:30:00
고흥 소록도,1,8.500000,남해,fgsj3,127.122800,34.504900,2025-03-17 10:30:00
구룡포 하정,1,10.500000,동해,fghe8,129.549700,35.960700,2025-03-17 10:30:00


In [83]:
df_Current.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
sta_nam_kor,String,69,42,0,강릉,3,null,null,강릉,완도 가교,해남 화산
obs_lay,String,69,3,0,1,42,null,null,1,1,3
wtr_tmp,Float,69,41,0,6.300000,4,8.156522,1.784202,4.900000,8.500000,11.900000
gru_nam,String,69,3,0,남해,35,null,null,남해,남해,서해
sta_cde,String,69,42,0,bgna3,3,null,null,bgj8a,fjh5a,wn087
lon,Double,69,42,0,128.949200,3,127.639399,1.190967,124.729500,127.708000,129.549700
lat,Double,69,42,0,37.799000,3,35.565158,1.387742,33.310400,34.808200,38.368100
obs_datetime,String,69,1,0,2025-03-17 10:30:00,69,null,null,2025-03-17 10:30:00,2025-03-17 10:30:00,2025-03-17 10:30:00


In [84]:
df_Current
    .filter { gru_nam.equals("동해") }
    .plot{
        layout{
            title = "Sea Water Quality"
            size = 1000 to 400
        }
        bars{
            alpha = 0.5
            x("sta_nam_kor"){
                axis{
                    name = "관측지점"
                }
            }
            y("wtr_tmp"){
                scale = continuous(0.0..15.0)
                axis {
                    name = "온도 °C"
                }
            }
            fillColor("obs_lay"){
                scale = categorical(
                    listOf(Color.BLUE, Color.GREEN, Color.RED),
                    listOf( "3", "2", "1"),
                )
                legend{
                    name= "관측 수심"
                    breaksLabeled("1" to "표층", "2" to "중층", "3" to "저층")
                }
            }

        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="uEgdwD"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Sea Water Quality"
},
"mapping":{
},
"data":{
"wtr_tmp":[6.300000190734863,6.300000190734863,6.199999809265137,5.5,5.199999809265137,5.099999904632568,10.5,11.699999809265137,11.800000190734863,11.800000190734863,7.599999904632568,7.5,6.900000095367432,6.699999809265137,6.300000190734863,6.0,10.899999618530273,10.800000190734863,10.800000190734863],
"obs_lay":["1","2","3","1","2","3","1","1","2","3","1","2","3","1","2","3","1","2","3"],
"sta_nam_kor":["강릉","강릉","강릉","고성 가진","고성 가진","고성 가진","구룡포 하정","기장","기장","기장","삼척","삼척","삼척","양양","양양","양양","영덕","영덕","영덕"]
},
"ggsize":{
"width":1000.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true,
"name":"관측지점"
},{
"aesthetic":"y",
"name":"온도 °C",
"limits":[0.0,15.0]
},{
"aesthetic":"fill",
"breaks":["1","2","3"],
"values":["#5470c6","#3ba272","#ee6666"],
"name":"관측 수심",
"limits":["3","2","1"],
"labels":["표층","중층","저층"]
}],
"layers":[{
"mapping":{
"x":"sta_nam_kor",
"y":"wtr_tmp",
"fill":"obs_lay"
},
"stat":"identity",
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"float",
"column":"wtr_tmp"
},{
"type":"str",
"column":"obs_lay"
}]
},
"spec_id":"44"
};
 var containerDiv = document.getElementById("uEgdwD");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1000.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 강릉 
 
 
 
 
 
 
 
 
 고성 가진 
 
 
 
 
 
 
 
 
 구룡포 하정 
 
 
 
 
 
 
 
 
 기장 
 
 
 
 
 
 
 
 
 삼척 
 
 
 
 
 
 
 
 
 양양 
 
 
 
 
 
 
 
 
 영덕 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 14 
 
 
 
 
 
 
 
 
 Sea Water Quality 
 
 
 
 
 온도 °C 
 
 
 
 
 관측지점 
 
 
 
 
 
 
 
 
 관측 수심 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 저층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 중층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 표층

In [96]:
df_Current
    .filter { gru_nam.equals("남해") }
    .plot{
        layout{
            title = "Sea Water Quality"
            size = 1000 to 400
        }
        bars{
            alpha = 0.5
            x("sta_nam_kor"){
                axis{
                    name = "관측지점"
                }
            }
            y("wtr_tmp"){
                scale = continuous(0.0..15.0)
                axis {
                    name = "온도 °C"
                }
            }
            fillColor("obs_lay"){
                scale = categorical(
                    listOf(Color.BLUE, Color.GREEN, Color.RED),
                    listOf( "3", "2", "1"),
                )
                legend{
                    name= "관측 수심"
                    breaksLabeled("1" to "표층", "2" to "중층", "3" to "저층")
                }
            }

        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="dJBbj3"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Sea Water Quality"
},
"mapping":{
},
"data":{
"wtr_tmp":[9.100000381469727,10.699999809265137,8.5,8.300000190734863,8.800000190734863,8.800000190734863,11.899999618530273,8.199999809265137,8.699999809265137,8.600000381469727,8.800000190734863,8.800000190734863,8.399999618530273,9.100000381469727,9.100000381469727,8.600000381469727,8.600000381469727,8.399999618530273,8.300000190734863,8.899999618530273,8.5,8.699999809265137,8.699999809265137,9.5,9.300000190734863,8.899999618530273,9.300000190734863,9.300000190734863,9.399999618530273,9.199999809265137,9.600000381469727,9.600000381469727,9.600000381469727,7.800000190734863,7.699999809265137],
"obs_lay":["1","1","1","1","1","2","1","1","1","2","1","1","1","1","2","1","2","1","2","1","1","1","2","1","2","1","1","1","1","2","1","2","3","1","2"],
"sta_nam_kor":["거제 가배","거제 일운","고흥 소록도","남해 강진","남해 미조","남해 미조","서제주","여수 신월","완도 가교","완도 가교","완도 감목","완도 금일","완도 노화도","완도 동백","완도 동백","완도 망남","완도 망남","완도 백도","완도 백도","완도 일정","완도 청산","장흥 회진","장흥 회진","통영 비산도","통영 비산도","통영 사량","통영 수월","통영 영운","통영 풍화","통영 풍화","통영 학림","통영 학림","통영 학림","해남 화산","해남 화산"]
},
"ggsize":{
"width":1000.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true,
"name":"관측지점"
},{
"aesthetic":"y",
"name":"온도 °C",
"limits":[0.0,15.0]
},{
"aesthetic":"fill",
"breaks":["1","2","3"],
"values":["#5470c6","#3ba272","#ee6666"],
"name":"관측 수심",
"limits":["3","2","1"],
"labels":["표층","중층","저층"]
}],
"layers":[{
"mapping":{
"x":"sta_nam_kor",
"y":"wtr_tmp",
"fill":"obs_lay"
},
"stat":"identity",
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"float",
"column":"wtr_tmp"
},{
"type":"str",
"column":"obs_lay"
}]
},
"spec_id":"53"
};
 var containerDiv = document.getElementById("dJBbj3");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1000.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 거제 가배 
 
 
 
 
 
 
 
 
 거제 일운 
 
 
 
 
 
 
 
 
 고흥 소록도 
 
 
 
 
 
 
 
 
 남해 강진 
 
 
 
 
 
 
 
 
 남해 미조 
 
 
 
 
 
 
 
 
 서제주 
 
 
 
 
 
 
 
 
 여수 신월 
 
 
 
 
 
 
 
 
 완도 가교 
 
 
 
 
 
 
 
 
 완도 감목 
 
 
 
 
 
 
 
 
 완도 금일 
 
 
 
 
 
 
 
 
 완도 노화도 
 
 
 
 
 
 
 
 
 완도 동백 
 
 
 
 
 
 
 
 
 완도 망남 
 
 
 
 
 
 
 
 
 완도 백도 
 
 
 
 
 
 
 
 
 완도 일정 
 
 
 
 
 
 
 
 
 완도 청산 
 
 
 
 
 
 
 
 
 장흥 회진 
 
 
 
 
 
 
 
 
 통영 비산도 
 
 
 
 
 
 
 
 
 통영 사량 
 
 
 
 
 
 
 
 
 통영 수월 
 
 
 
 
 
 
 
 
 통영 영운 
 
 
 
 
 
 
 
 
 통영 풍화 
 
 
 
 
 
 
 
 
 통영 학림 
 
 
 
 
 
 
 
 
 해남 화산 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 14 
 
 
 
 
 
 
 
 
 Sea Water Quality 
 
 
 
 
 온도 °C 
 
 
 
 
 관측지점 
 
 
 
 
 
 
 
 
 관측 수심 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 저층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 중층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 표층

In [97]:
df_Current
    .filter { gru_nam.equals("서해") }
    .plot{
        layout{
            title = "Sea Water Quality"
            size = 1000 to 400
        }
        bars{
            alpha = 0.5
            x("sta_nam_kor"){
                axis{
                    name = "관측지점"
                }
            }
            y("wtr_tmp"){
                scale = continuous(0.0..15.0)
                axis {
                    name = "온도 °C"
                }
            }
            fillColor("obs_lay"){
                scale = categorical(
                    listOf(Color.BLUE, Color.GREEN, Color.RED),
                    listOf( "3", "2", "1"),
                )
                legend{
                    name= "관측 수심"
                    breaksLabeled("1" to "표층", "2" to "중층", "3" to "저층")
                }
            }

        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="LTJ5GM"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Sea Water Quality"
},
"mapping":{
},
"data":{
"wtr_tmp":[6.199999809265137,7.400000095367432,4.900000095367432,5.699999809265137,6.400000095367432,6.300000190734863,7.099999904632568,7.400000095367432,7.400000095367432,5.800000190734863,6.199999809265137,6.199999809265137,5.400000095367432,5.300000190734863,7.5],
"obs_lay":["1","1","1","1","1","2","1","1","2","1","1","2","1","2","1"],
"sta_nam_kor":["군산 신시도","목포","백령도","서산 지곡","서산 창리","서산 창리","신안 압해","진도 전두","진도 전두","태안 고남","태안 대야도","태안 대야도","태안 파도리","태안 파도리","해남 임하"]
},
"ggsize":{
"width":1000.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true,
"name":"관측지점"
},{
"aesthetic":"y",
"name":"온도 °C",
"limits":[0.0,15.0]
},{
"aesthetic":"fill",
"breaks":["1","2","3"],
"values":["#5470c6","#3ba272","#ee6666"],
"name":"관측 수심",
"limits":["3","2","1"],
"labels":["표층","중층","저층"]
}],
"layers":[{
"mapping":{
"x":"sta_nam_kor",
"y":"wtr_tmp",
"fill":"obs_lay"
},
"stat":"identity",
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"sta_nam_kor"
},{
"type":"float",
"column":"wtr_tmp"
},{
"type":"str",
"column":"obs_lay"
}]
},
"spec_id":"56"
};
 var containerDiv = document.getElementById("LTJ5GM");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1000.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 군산 신시도 
 
 
 
 
 
 
 
 
 목포 
 
 
 
 
 
 
 
 
 백령도 
 
 
 
 
 
 
 
 
 서산 지곡 
 
 
 
 
 
 
 
 
 서산 창리 
 
 
 
 
 
 
 
 
 신안 압해 
 
 
 
 
 
 
 
 
 진도 전두 
 
 
 
 
 
 
 
 
 태안 고남 
 
 
 
 
 
 
 
 
 태안 대야도 
 
 
 
 
 
 
 
 
 태안 파도리 
 
 
 
 
 
 
 
 
 해남 임하 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 14 
 
 
 
 
 
 
 
 
 Sea Water Quality 
 
 
 
 
 온도 °C 
 
 
 
 
 관측지점 
 
 
 
 
 
 
 
 
 관측 수심 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 저층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 중층 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 표층

In [85]:
fun<T> makePointGeoJSON( coordinates:List<Pair<Double, Double>>, properties:Map<String, List<T>>): String {
    val first_str = "{" + "\n" +
            "\t\"type\": \"FeatureCollection\"," + "\n" +
            "\t\"features\": [" + "\n"

    val end_str = "\t]" + "\n" +
            "}" + "\n"

    var features_str = ""

    coordinates.forEachIndexed { index,    pair ->

        val delimiter1 = if (index < coordinates.size - 1) "," else ""

        features_str += "\t\t{\n" +
                "\t\t\t\"type\": \"Feature\",\n" +
                "\t\t\t\"geometry\": {\n" +
                "\t\t\t\t\"type\": \"Point\",\n" +
                "\t\t\t\t\"coordinates\": [${pair.first}, ${pair.second}]\n" +
                "\t\t\t},\n" +
                "\t\t\t\"properties\": {\n"

        val propertieIterator = properties.entries.iterator()

        while (propertieIterator.hasNext()) {
            val (key, values) = propertieIterator.next()
            val delimiter2 = if (propertieIterator.hasNext()) "," else ""

            features_str += "\t\t\t\t\"${key}\": \"${values[index]}\"${delimiter2} \n"
        }

        features_str +=  "\t\t\t}\n" + "\t\t}${delimiter1}\n"

    }

    return first_str + features_str + end_str
}

In [86]:
df_Current.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
sta_nam_kor,String,69,42,0,강릉,3,null,null,강릉,완도 가교,해남 화산
obs_lay,String,69,3,0,1,42,null,null,1,1,3
wtr_tmp,Float,69,41,0,6.300000,4,8.156522,1.784202,4.900000,8.500000,11.900000
gru_nam,String,69,3,0,남해,35,null,null,남해,남해,서해
sta_cde,String,69,42,0,bgna3,3,null,null,bgj8a,fjh5a,wn087
lon,Double,69,42,0,128.949200,3,127.639399,1.190967,124.729500,127.708000,129.549700
lat,Double,69,42,0,37.799000,3,35.565158,1.387742,33.310400,34.808200,38.368100
obs_datetime,String,69,1,0,2025-03-17 10:30:00,69,null,null,2025-03-17 10:30:00,2025-03-17 10:30:00,2025-03-17 10:30:00


In [81]:
val coordinates = df_Current.filter {
    obs_lay.equals("1")
}.select{
   sta_cde and sta_nam_kor and  lon and lat
}.sortBy{sta_cde and  sta_nam_kor  }

coordinates

sta_cde,sta_nam_kor,lon,lat
bgj8a,기장,129.227000,35.187000
bgna3,강릉,128.949200,37.799000
br001,태안 고남,126.433300,36.415800
bsc87,삼척,129.312700,37.302300
byd8a,영덕,129.437000,36.573700
byy87,양양,128.699800,38.080800
egsi4,군산 신시도,126.442200,35.816800
ejhfc,장흥 회진,126.980900,34.469900
ejj47,서제주,126.164000,33.310400
emp67,목포,126.365300,34.789200


In [ ]:
val points = df_Current.filter {
    obs_lay.equals("1")
}.select{
    sta_cde and sta_nam_kor and  lon and lat
}.sortBy{sta_cde and  sta_nam_kor
}.map { Pair(lon, lat) }.toList()


In [ ]:
val properties = df_Current.filter {
   obs_lay.equals("1")
}
    .add("tempBoundary"){
        when(wtr_tmp.toFloat()) {
            in 0.0..4.9 -> "Low"
            in 5.0..7.9 -> "Middle"
            in 8.0..15.9 -> "High"
            else -> "Unknown"
        }
    }
    .select {sta_cde and  sta_nam_kor and wtr_tmp and obs_datetime  and "tempBoundary"}
    .sortBy{sta_cde and  sta_nam_kor  }.toMap()


In [104]:
DataFrame.readJsonStr(makePointGeoJSON(points, properties))
    .writeJson("/Volumes/WorkSpace/Notebook/data/nifsPoint.json")

In [105]:
val southKorea = GeoDataFrame.readGeoJson("/Volumes/WorkSpace/Notebook/data/southkorea.geojson")
val staPoint = GeoDataFrame.readGeoJson("/Volumes/WorkSpace/Notebook/data/nifsPoint.json")

In [106]:
val southKoreaBounds: Envelope = southKorea.bounds().also {
    // Use JTS API for in-place envelope expansion
    it.expandBy(0.2)
}

In [107]:

southKorea.plot{
    layout{
        title = "Korea Sea Water Quality"
        size = 1400 to 1400
        //  theme = Theme.HIGH_CONTRAST_DARK
        style(Style.BW) {
            this.panel.background {
                fillColor =  Color.hex("#537ab5")
                borderLineColor = Color.hex("#EFC623")
                borderLineWidth = 6.0
            }
        }
    }

    geoMap{
        fillColor= Color.hex("#e5f5e0")
        borderLine {
            width = 0.5
            color = Color.hex("#a1d99b")
        }
    }

    withData(staPoint) {

        geoPoints {
            size = 5.0
            symbol = Symbol.CIRCLE_FILLED
            fillColor(tempBoundary)
            tooltips(title = value(sta_nam_kor)){
                line("수온 ${value(wtr_tmp)}°C")
                line("측정시간", value(obs_datetime))
            }
        }
    }


}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="YUAI2f"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Korea Sea Water Quality"
},
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"ggsize":{
"width":1400.0,
"height":1400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"size":0.5,
"color":"#a1d99b",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"fill":"#e5f5e0",
"map":{
"geometry":["{\"type\":\"MultiPolygon\",\"coordinates\":[[[[128.3649194917,38.6243350685],[128.3947835223,38.5780740365],[128.4414167978,38.5058046474],[128.4506941248,38.4741070283],[128.4744572914,38.4260929476],[128.5627547951,38.2889672584],[128.6316023609,38.1449648288],[128.6428328922,38.1292992431],[128.6787215922,38.0943058017],[128.6924748566,38.0649275611],[128.8591414717,37.8767764335],[128.9238386781,37.8023134422],[129.0013126278,37.7343609539],[129.0122990573,37.7272403587],[129.0642195107,37.6788596209],[129.0669051816,37.6584333992],[129.0671492481,37.6338564489],[129.0733342121,37.6130232322],[129.1087345365,37.5973981199],[129.1189884678,37.5806338402],[129.1482039365,37.505112962],[129.1612247887,37.4876162483],[129.1768497994,37.4801699777],[129.1899519893,37.46938706],[129.2354435897,37.3982608671],[129.2627872718,37.370917],[129.2722274533,37.3558617958],[129.2758894805,37.3399111335],[129.2828068399,37.324448965],[129.3310653374,37.2821720069],[129.3464461534,37.2475039615],[129.3601180378,37.1717796949],[129.3789168242,37.1301129554],[129.4301863659,37.0730654479],[129.434092682,37.0611026724],[129.4282332864,37.0495466474],[129.4189559483,37.0385196388],[129.4130965445,37.0277367114],[129.412689633,37.018052448],[129.4199324954,36.9867617989],[129.4287215264,36.8974062755],[129.4379989231,36.8561466033],[129.4712019984,36.7718773096],[129.4738875737,36.7306175937],[129.463470895,36.6924502549],[129.4409285801,36.6577822907],[129.4226180537,36.6166446314],[129.4228621805,36.6149763138],[129.4279891043,36.5756696199],[129.4416609954,36.5346540236],[129.447113488,36.4933129127],[129.4409285498,36.4079449961],[129.4348250761,36.3895938351],[129.4039006176,36.3594425052],[129.3918563304,36.3424747029],[129.3865666203,36.322699319],[129.3823348485,36.201849707],[129.3857528032,36.1854515707],[129.3918563333,36.1763370125],[129.3985294907,36.1726748678],[129.4039005976,36.1676293183],[129.4096785794,36.1336123673],[129.4170841685,36.1075706822],[129.4251408289,36.1030134284],[129.4305119006,36.0973981528],[129.4267684105,36.0817731525],[129.4187118006,36.0729841455],[129.408295117,36.0689151043],[129.3984481031,36.066310922],[129.3817826673,36.0551746207],[129.3808587342,36.0414868754],[129.4019370429,36.0256600839],[129.4170806359,36.0119127132],[129.4330514795,35.9961077403],[129.4533314103,35.9932873685],[129.4790145222,36.0095889398],[129.5425724472,36.065904012],[129.5547949502,36.0824779008],[129.5712996819,36.0718448065],[129.574961804,36.0546736047],[129.5820418747,36.0368106353],[129.5872501971,36.016913155],[129.5716437692,35.9971243936],[129.5355895705,35.9390356641],[129.532839056,35.9130129127],[129.5221343282,35.8445522464],[129.4962165012,35.785073939],[129.497650567,35.747870147],[129.4711259106,35.6832223268],[129.4515757092,35.6518022926],[129.443053381,35.6360899906],[129.4513757628,35.6182433264],[129.4605899019,35.6092942117],[129.467275133,35.5996675541],[129.46